# Fare Prediction: Initial Verification and Setup

This notebook currently performs verification only. It does not re-clean the data, modify the chronological splits, select a final feature list, use the test set for development, or train a model.

## Verification scope

- Confirm the official train, validation, and test Parquet files and metadata row counts.
- Check pickup-timestamp coverage and chronological split boundaries.
- Confirm `base_fare` availability and inspect small train/validation samples only.
- Read the official feature contract and report allowed, conditional, forbidden, and missing-value guidance.

**STOP after this verification step. No model training is performed.**

In [ ]:
from pathlib import Path
import json
import pyarrow.parquet as pq
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "splits").is_dir())
SPLIT_DIR = ROOT / "data" / "splits"
CONTRACT_PATH = ROOT / "data" / "feature_contract.md"
REPORT_PATH = ROOT / "reports" / "data_cleaning_decisions.md"
HANDOVER_PATH = ROOT / "docs" / "MEMBER1_HANDOVER.md"
EXPECTED_ROWS = {
    "train": 31_988_176,
    "validation": 6_814_901,
    "test": 6_730_257,
}
TARGET = "base_fare"

print("## 1. Official input files and metadata")
split_metadata = {}
for split_name, expected_rows in EXPECTED_ROWS.items():
    split_path = SPLIT_DIR / f"{split_name}.parquet"
    if not split_path.exists():
        raise FileNotFoundError(split_path)
    parquet = pq.ParquetFile(split_path)
    actual_rows = parquet.metadata.num_rows
    split_metadata[split_name] = {
        "path": split_path,
        "rows": actual_rows,
        "columns": parquet.schema_arrow.names,
        "row_groups": parquet.metadata.num_row_groups,
        "file_size_mb": split_path.stat().st_size / 1_000_000,
    }
    print(f"{split_name}: rows={actual_rows:,}; row groups={parquet.metadata.num_row_groups}; size={split_metadata[split_name]['file_size_mb']:.3f} MB")
    assert actual_rows == expected_rows, f"Unexpected {split_name} row count"
    assert TARGET in parquet.schema_arrow.names, f"Missing {TARGET} in {split_name}"

all_rows = sum(metadata["rows"] for metadata in split_metadata.values())
print(f"Total rows: {all_rows:,}")
assert all_rows == 45_533_334

print("\n## 2. Chronological pickup-timestamp verification")
def scan_timestamps(split_name):
    parquet = pq.ParquetFile(split_metadata[split_name]["path"])
    start = None
    end = None
    previous_last = None
    internal_inversions = 0
    unique_dates = set()
    for row_group_index in range(parquet.metadata.num_row_groups):
        table = parquet.read_row_group(row_group_index, columns=["pickup_timestamp"])
        timestamps = pd.to_datetime(table["pickup_timestamp"].to_pandas(), errors="raise")
        if len(timestamps) == 0:
            continue
        internal_inversions += int((timestamps.iloc[1:].to_numpy() < timestamps.iloc[:-1].to_numpy()).sum())
        if previous_last is not None and timestamps.iloc[0] < previous_last:
            internal_inversions += 1
        previous_last = timestamps.iloc[-1]
        start = timestamps.min() if start is None or timestamps.min() < start else start
        end = timestamps.max() if end is None or timestamps.max() > end else end
        unique_dates.update(timestamps.dt.date.tolist())
    return {
        "start": start,
        "end": end,
        "unique_pickup_dates": len(unique_dates),
        "internal_inversions": internal_inversions,
    }

timestamp_summary = {name: scan_timestamps(name) for name in EXPECTED_ROWS}
for split_name, summary in timestamp_summary.items():
    print(f"{split_name}: {summary['start']} through {summary['end']}; unique dates={summary['unique_pickup_dates']}; within-file inversions={summary['internal_inversions']}")

assert timestamp_summary["train"]["end"] < timestamp_summary["validation"]["start"]
assert timestamp_summary["validation"]["end"] < timestamp_summary["test"]["start"]
assert timestamp_summary["test"]["end"] == max(summary["end"] for summary in timestamp_summary.values())
if any(summary["internal_inversions"] for summary in timestamp_summary.values()):
    print("WARNING: split files have nonzero within-file timestamp inversions; split boundaries are chronological, but physical row order is not globally sorted.")
else:
    print("Within-file timestamp order: PASS")
print("Chronological split boundaries: PASS")

print("\n## 3. Official feature-contract guidance")
for required_path in [CONTRACT_PATH, REPORT_PATH, HANDOVER_PATH]:
    assert required_path.exists(), required_path
contract = CONTRACT_PATH.read_text(encoding="utf-8")
contract_lower = contract.lower()
for phrase in ["base_fare", "trip_duration_minutes", "allowed pre-trip candidate features", "conditional features", "forbidden leakage fields", "missing-value notes"]:
    assert phrase in contract_lower, f"Missing contract section or target: {phrase}"

print("Fare target:", TARGET)
print("Candidate features are documented under 'Allowed pre-trip candidate features'.")
print("Conditional features are documented under 'Conditional features'.")
print("Forbidden leakage fields are documented under 'Forbidden leakage fields'.")
print("Missing-value guidance is documented under 'Missing-value notes'.")
print("No final feature list selected.")

print("\n## 4. Small train and validation samples")
sample_columns = ["pickup_timestamp", "base_fare", "provider_code", "origin_loc_id", "dest_loc_id", "pickup_hour", "day_of_week", "month", "weekend"]
for split_name in ["train", "validation"]:
    parquet = pq.ParquetFile(split_metadata[split_name]["path"])
    sample = parquet.read_row_group(0, columns=sample_columns).to_pandas().head(1_000)
    print(f"\n{split_name} sample shape: {sample.shape}")
    print("Sample columns:", sample.columns.tolist())
    print("base_fare sample summary:")
    print(sample["base_fare"].describe().to_string())
    print("Sample missing values:")
    print(sample.isna().sum().to_dict())

print("\n## 5. Verification status")
print("train split: PASS")
print("validation split: PASS")
print("test split: PASS")
print("chronological ordering: PASS (split boundaries; physical row order warning above if applicable)")
print("base_fare availability: PASS")
print("feature contract availability: PASS")
print("leakage guidance: PASS")
print("WARNING: statistics above are metadata and small-sample checks; no full-data target summary was computed.")
print("STOP: Fare prediction verification only. No feature selection or model training was performed.")

## Step 2: Final first-pass features for fare prediction

The fare target is `base_fare`. This deliberately simple first-pass candidate set contains only seven features. It keeps the core pickup-time, provider, and origin/destination signals while avoiding redundant or high-cardinality additions. Extra features can be evaluated later only if validation metrics improve.

### Final first-pass candidate lists

- **Numeric features:** `pickup_hour`, `month`
- **Categorical features:** `provider_code`, `day_of_week`, `weekend`, `origin_loc_id`, `dest_loc_id`
- **Target:** `base_fare`
- **Total features:** 7

The following are intentionally excluded from the first-pass set:

- `pickup_date`: high-cardinality and largely overlaps with the existing time features.
- `route_id`: duplicates the origin/destination information.
- `pickup_zone_name`, `dropoff_zone_name`, `pickup_borough_name`, `dropoff_borough_name`: duplicate the location IDs and add unnecessary high-cardinality categorical detail for the first model.

The following conditional features remain excluded: `distance_miles`, `rider_count`, `rate_class_id`, `fare_settlement_method`, and `offline_record_flag`. In particular, `distance_miles` may be used only if the team confirms that it is an estimated route distance available before trip start.

All leakage exclusions remain unchanged: never use `base_fare`, `trip_duration_minutes`, `speed_mph`, `dropoff_timestamp`, tips, final charges, post-trip fees, raw unrestricted timestamps, audit outcomes, or source/provenance fields as fare inputs. No preprocessing, model training, tuning, or test-data analysis is performed in Step 2.

In [ ]:
## 1. Define the final first-pass candidate variables
numeric_features = [
    "pickup_hour",
    "month",
]

categorical_features = [
    "provider_code",
    "day_of_week",
    "weekend",
    "origin_loc_id",
    "dest_loc_id",
]
target = "base_fare"

print("Final first-pass numeric features:", numeric_features)
print("Final first-pass categorical features:", categorical_features)
print("Target:", target)
print("Total first-pass features:", len(numeric_features) + len(categorical_features))
assert len(numeric_features) + len(categorical_features) == 7

## 2. Validate the simplified candidate names
train_path = ROOT / "data" / "splits" / "train.parquet"
validation_path = ROOT / "data" / "splits" / "validation.parquet"
train_schema = pq.ParquetFile(train_path).schema_arrow
validation_schema = pq.ParquetFile(validation_path).schema_arrow
train_columns = set(train_schema.names)
validation_columns = set(validation_schema.names)
all_candidates = numeric_features + categorical_features
assert set(all_candidates).issubset(train_columns)
assert set(all_candidates).issubset(validation_columns)
assert target in train_columns and target in validation_columns

excluded_first_pass = [
    "pickup_date",
    "route_id",
    "pickup_zone_name",
    "dropoff_zone_name",
    "pickup_borough_name",
    "dropoff_borough_name",
]
conditional_features = [
    "distance_miles",
    "rider_count",
    "rate_class_id",
    "fare_settlement_method",
    "offline_record_flag",
]
for column in excluded_first_pass + conditional_features:
    assert column not in all_candidates

forbidden_fields = [
    "base_fare",
    "trip_duration_minutes",
    "speed_mph",
    "dropoff_timestamp",
    "charge_total",
    "driver_tip_payment",
    "toll_total",
    "surcharge_misc",
    "transit_tax",
    "service_improvement_fee",
    "zone_congestion_fee",
    "Airport_fee",
    "congestion_relief_fee",
    "pickup_timestamp",
    "source_file",
    "source_month",
    "source_row_1based",
    "audit_zero_distance_nonzero_fare",
    "audit_zero_riders",
    "audit_speed_80_to_100",
    "audit_pickup_outside_source_month",
    "audit_dropoff_outside_source_month",
    "audit_boundary_category",
    "audit_pickup_incomplete_zone_labels",
    "audit_dropoff_incomplete_zone_labels",
]
assert not set(all_candidates).intersection(forbidden_fields)
print("Simplified candidate schema validation: PASS")
print("Excluded first-pass redundant/high-cardinality fields:", excluded_first_pass)
print("Conditional features excluded:", conditional_features)
print("Forbidden leakage fields excluded: PASS")
print("No preprocessing or model training performed.")
print("STOP: Step 2 first-pass feature definition only.")

## Step 3: Preprocessing verification

For this verification, numeric missing values use median imputation. Categorical missing values use most-frequent imputation, followed by `OneHotEncoder(handle_unknown="ignore")` so validation categories not observed in the small training sample do not cause transformation errors.

The preprocessor is structured as a `ColumnTransformer` so it can later be placed inside the final model `Pipeline`. Fitting preprocessing inside that future Pipeline on training data will prevent preprocessing leakage. Here, fitting is deliberately limited to a small training sample; no model is trained and the test split is not read.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

print("## Step 3 preprocessing verification")

selected_features = numeric_features + categorical_features
train_path = ROOT / "data" / "splits" / "train.parquet"
validation_path = ROOT / "data" / "splits" / "validation.parquet"

train_schema = pq.ParquetFile(train_path).schema_arrow
validation_schema = pq.ParquetFile(validation_path).schema_arrow
assert set(selected_features).issubset(train_schema.names)
assert set(selected_features).issubset(validation_schema.names)
print("Selected features exist in train and validation schemas: PASS")

sample_columns = selected_features + [target]
train_table = pq.ParquetFile(train_path).read_row_group(0, columns=sample_columns)
validation_table = pq.ParquetFile(validation_path).read_row_group(0, columns=sample_columns)
train_sample = train_table.to_pandas().head(1_000)
validation_sample = validation_table.to_pandas().head(1_000)

print("Train sample missingness:")
print(train_sample[selected_features].isna().sum().to_dict())
print("Validation sample missingness:")
print(validation_sample[selected_features].isna().sum().to_dict())

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features),
])

# Fit preprocessing only on the small training sample.
X_train_sample = train_sample[selected_features]
X_validation_sample = validation_sample[selected_features]
preprocessor.fit(X_train_sample)
X_train_transformed = preprocessor.transform(X_train_sample)
X_validation_transformed = preprocessor.transform(X_validation_sample)

print("Original training sample shape:", X_train_sample.shape)
print("Original validation sample shape:", X_validation_sample.shape)
print("Transformed training shape:", X_train_transformed.shape)
print("Transformed validation shape:", X_validation_transformed.shape)
print("Transformed feature count:", X_train_transformed.shape[1])
print("Validation transformation without errors: PASS")
print("Preprocessing status: PASS")
print("No test data read; no model trained; no tuning performed.")
print("STOP: Step 3 preprocessing verification only.")

## Step 4: Median baseline for fare prediction

The median baseline is the minimum benchmark for `base_fare`: it predicts the same training-target median for every validation row. Later machine-learning models must meaningfully outperform this baseline on validation metrics.

This verification reads only the `base_fare` column from train and validation, processes Parquet row groups incrementally, and stores temporary numeric vectors on disk for exact medians. The test split is not read, and no preprocessing pipeline is needed for a constant prediction.

In [ ]:
from pathlib import Path
import math
import tempfile
import numpy as np
import pyarrow.parquet as pq
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("## Step 4 median baseline verification")

TARGET = "base_fare"
TRAIN_PATH = ROOT / "data" / "splits" / "train.parquet"
VALIDATION_PATH = ROOT / "data" / "splits" / "validation.parquet"
EXPECTED_VALIDATION_ROWS = 6_814_901


def append_target_values(parquet_path, output_handle):
    parquet = pq.ParquetFile(parquet_path)
    count = 0
    for row_group_index in range(parquet.metadata.num_row_groups):
        table = parquet.read_row_group(row_group_index, columns=[TARGET])
        values = table[TARGET].to_numpy(zero_copy_only=False).astype("float64", copy=False)
        values = values[np.isfinite(values)]
        values.tofile(output_handle)
        count += len(values)
    return count


def exact_median(binary_path, count):
    values = np.memmap(binary_path, dtype="float64", mode="r+", shape=(count,))
    try:
        middle = count // 2
        if count % 2:
            values.partition(middle)
            result = float(values[middle])
        else:
            values.partition((middle - 1, middle))
            result = (float(values[middle - 1]) + float(values[middle])) / 2
        values.flush()
        return result
    finally:
        values._mmap.close()


def validation_metrics(parquet_path, prediction, median_output):
    parquet = pq.ParquetFile(parquet_path)
    row_count = 0
    absolute_error_sum = 0.0
    squared_error_sum = 0.0
    target_sum = 0.0
    target_squared_sum = 0.0
    with median_output:
        for row_group_index in range(parquet.metadata.num_row_groups):
            table = parquet.read_row_group(row_group_index, columns=[TARGET])
            values = table[TARGET].to_numpy(zero_copy_only=False).astype("float64", copy=False)
            values = values[np.isfinite(values)]
            errors = values - prediction
            absolute_error_sum += float(np.abs(errors).sum())
            squared_error_sum += float(np.square(errors).sum())
            target_sum += float(values.sum())
            target_squared_sum += float(np.square(values).sum())
            values.tofile(median_output)
            row_count += len(values)
    mean_target = target_sum / row_count
    total_squared_sum = target_squared_sum - row_count * mean_target**2
    return {
        "rows": row_count,
        "mae": absolute_error_sum / row_count,
        "rmse": math.sqrt(squared_error_sum / row_count),
        "r2": 1 - squared_error_sum / total_squared_sum,
        "mean": mean_target,
    }

with tempfile.TemporaryDirectory(prefix="fare_median_baseline_") as temporary_directory:
    temporary_directory = Path(temporary_directory)
    training_values_path = temporary_directory / "training_base_fare.bin"
    validation_values_path = temporary_directory / "validation_base_fare.bin"

    with training_values_path.open("wb") as training_output:
        training_count = append_target_values(TRAIN_PATH, training_output)
    training_median = exact_median(training_values_path, training_count)

    with validation_values_path.open("wb") as validation_output:
        validation_summary = validation_metrics(VALIDATION_PATH, training_median, validation_output)
    validation_median = exact_median(validation_values_path, validation_summary["rows"])

assert validation_summary["rows"] == EXPECTED_VALIDATION_ROWS
baseline_results = {
    "Model": "Median baseline",
    "MAE": validation_summary["mae"],
    "RMSE": validation_summary["rmse"],
    "R2": validation_summary["r2"],
    "Validation Rows": validation_summary["rows"],
}

print("Training fare median:", training_median)
print("Validation target mean:", validation_summary["mean"])
print("Validation target median:", validation_median)
print("Validation rows evaluated:", validation_summary["rows"])
print("\nBaseline results:")
print("| Model | MAE | RMSE | R² | Validation Rows |")
print("|---|---:|---:|---:|---:|")
print(f"| {baseline_results['Model']} | {baseline_results['MAE']:.6f} | {baseline_results['RMSE']:.6f} | {baseline_results['R2']:.6f} | {baseline_results['Validation Rows']:,} |")
print("\nPASS: median baseline evaluated on validation using training target median only.")
print("WARNING: no test data was read; no stronger model or tuning was run.")
print("STOP: Step 4 median baseline only.")

## Step 5: LinearRegression fare model

This first model uses the finalized seven-feature set and the Step 3 preprocessing pipeline inside one scikit-learn `Pipeline`. Because the full training matrix is large, fitting uses a deterministic, evenly spaced sample of up to 2,500 rows from each training Parquet row group. This is a representative training subset, not the full training split; the limitation is recorded with the results.

Validation is evaluated row-group by row-group using the fitted pipeline. The test split is not read, no tuning is performed, and no RandomForest or Gradient Boosting model is trained.

In [ ]:
import time
import pyarrow as pa
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("## Step 5 LinearRegression verification")

MODEL_FEATURES = numeric_features + categorical_features
TARGET = "base_fare"
TRAIN_PATH = ROOT / "data" / "splits" / "train.parquet"
VALIDATION_PATH = ROOT / "data" / "splits" / "validation.parquet"
SAMPLE_PER_ROW_GROUP = 2_500
EXPECTED_VALIDATION_ROWS = 6_814_901

train_parquet = pq.ParquetFile(TRAIN_PATH)
validation_parquet = pq.ParquetFile(VALIDATION_PATH)
assert set(MODEL_FEATURES + [TARGET]).issubset(train_parquet.schema_arrow.names)
assert set(MODEL_FEATURES + [TARGET]).issubset(validation_parquet.schema_arrow.names)

# Build a deterministic, evenly spaced sample across every training row group.
training_sample_parts = []
for row_group_index in range(train_parquet.metadata.num_row_groups):
    table = train_parquet.read_row_group(row_group_index, columns=MODEL_FEATURES + [TARGET])
    row_count = table.num_rows
    sample_size = min(SAMPLE_PER_ROW_GROUP, row_count)
    indices = np.linspace(0, row_count - 1, num=sample_size, dtype=np.int64)
    training_sample_parts.append(table.take(pa.array(indices)).to_pandas())
training_sample = pd.concat(training_sample_parts, ignore_index=True)
X_train_sample = training_sample[MODEL_FEATURES]
y_train_sample = training_sample[TARGET].to_numpy(dtype="float64")

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

fit_start = time.perf_counter()
pipeline.fit(X_train_sample, y_train_sample)
training_time_seconds = time.perf_counter() - fit_start

# Stream validation row groups so predictions are never accumulated in memory.
validation_rows = 0
absolute_error_sum = 0.0
squared_error_sum = 0.0
target_sum = 0.0
target_squared_sum = 0.0
for row_group_index in range(validation_parquet.metadata.num_row_groups):
    table = validation_parquet.read_row_group(row_group_index, columns=MODEL_FEATURES + [TARGET])
    validation_frame = table.to_pandas()
    X_validation = validation_frame[MODEL_FEATURES]
    y_validation = validation_frame[TARGET].to_numpy(dtype="float64")
    predictions = pipeline.predict(X_validation)
    errors = predictions - y_validation
    validation_rows += len(y_validation)
    absolute_error_sum += float(np.abs(errors).sum())
    squared_error_sum += float(np.square(errors).sum())
    target_sum += float(y_validation.sum())
    target_squared_sum += float(np.square(y_validation).sum())

validation_mean = target_sum / validation_rows
total_sum_of_squares = target_squared_sum - validation_rows * validation_mean**2
linear_regression_mae = absolute_error_sum / validation_rows
linear_regression_rmse = float(np.sqrt(squared_error_sum / validation_rows))
linear_regression_r2 = 1 - squared_error_sum / total_sum_of_squares

assert validation_rows == EXPECTED_VALIDATION_ROWS
transformed_feature_count = pipeline.named_steps["preprocessor"].transform(X_train_sample.iloc[:1]).shape[1]
results = {
    "Model": "LinearRegression",
    "Train Rows": len(training_sample),
    "Validation Rows": validation_rows,
    "MAE": linear_regression_mae,
    "RMSE": linear_regression_rmse,
    "R2": linear_regression_r2,
    "Training Time": training_time_seconds,
}

baseline_mae = 11.713890
baseline_rmse = 19.724010
baseline_r2 = -0.148117
print("Training rows used:", results["Train Rows"])
print("Validation rows evaluated:", results["Validation Rows"])
print("Training time (seconds):", f"{training_time_seconds:.3f}")
print("Transformed feature count:", transformed_feature_count)
print("\n| Model | Train Rows | Validation Rows | MAE | RMSE | R² | Training Time |")
print("|---|---:|---:|---:|---:|---:|---:|")
print(f"| {results['Model']} | {results['Train Rows']:,} | {results['Validation Rows']:,} | {results['MAE']:.6f} | {results['RMSE']:.6f} | {results['R2']:.6f} | {results['Training Time']:.3f}s |")
print("\nMedian baseline comparison:")
print(f"MAE change (LinearRegression - baseline): {linear_regression_mae - baseline_mae:.6f}")
print(f"RMSE change (LinearRegression - baseline): {linear_regression_rmse - baseline_rmse:.6f}")
print(f"R² change (LinearRegression - baseline): {linear_regression_r2 - baseline_r2:.6f}")
print("Beats median baseline on all three metrics:", linear_regression_mae < baseline_mae and linear_regression_rmse < baseline_rmse and linear_regression_r2 > baseline_r2)
print("PASS: LinearRegression evaluated on validation only.")
print("WARNING: training used a deterministic sampled subset; results are not full-training-fit results.")
print("STOP: Step 5 LinearRegression only. No test data, tuning, RandomForest, or Gradient Boosting used.")

## Step 6: RandomForestRegressor comparison

This step compares a first-pass Random Forest with the existing median baseline and sampled LinearRegression result. It uses the same deterministic 445,000-row training sample for a fair comparison, keeps the Step 3 preprocessing inside one scikit-learn `Pipeline`, and evaluates only on validation.

The initial CPU-conscious configuration uses 100 trees, `max_depth=18`, `min_samples_leaf=5`, `max_features=0.7`, `random_state=42`, and `n_jobs=-1`. These settings are a fixed first pass, not tuning. The test split is not read.

### Interpretation

With this fixed configuration, RandomForestRegressor does not beat LinearRegression: MAE is 9.209341 versus 8.689478, RMSE is 14.128331 versus 13.928211, and R² is 0.410915 versus 0.427485. It still clearly beats the median baseline. The current result does not justify the added training cost and complexity for this first pass, although another nonlinear candidate may still be tested later under the same leakage-safe evaluation protocol. No tuning is performed in Step 6.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

print("## Step 6 RandomForestRegressor verification")

MODEL_FEATURES = numeric_features + categorical_features
TARGET = "base_fare"
TRAIN_PATH = ROOT / "data" / "splits" / "train.parquet"
VALIDATION_PATH = ROOT / "data" / "splits" / "validation.parquet"
SAMPLE_PER_ROW_GROUP = 2_500
EXPECTED_TRAIN_SAMPLE_ROWS = 445_000
EXPECTED_VALIDATION_ROWS = 6_814_901

train_parquet = pq.ParquetFile(TRAIN_PATH)
validation_parquet = pq.ParquetFile(VALIDATION_PATH)
assert set(MODEL_FEATURES + [TARGET]).issubset(train_parquet.schema_arrow.names)
assert set(MODEL_FEATURES + [TARGET]).issubset(validation_parquet.schema_arrow.names)

# Recreate the same deterministic sample used by Step 5: 2,500 rows per train row group.
training_sample_parts = []
for row_group_index in range(train_parquet.metadata.num_row_groups):
    table = train_parquet.read_row_group(row_group_index, columns=MODEL_FEATURES + [TARGET])
    row_count = table.num_rows
    sample_size = min(SAMPLE_PER_ROW_GROUP, row_count)
    indices = np.linspace(0, row_count - 1, num=sample_size, dtype=np.int64)
    training_sample_parts.append(table.take(pa.array(indices)).to_pandas())
training_sample = pd.concat(training_sample_parts, ignore_index=True)
X_train_sample = training_sample[MODEL_FEATURES]
y_train_sample = training_sample[TARGET].to_numpy(dtype="float64")
assert len(training_sample) == EXPECTED_TRAIN_SAMPLE_ROWS

random_forest = RandomForestRegressor(
    n_estimators=100,
    max_depth=18,
    min_samples_leaf=5,
    max_features=0.7,
    random_state=42,
    n_jobs=-1,
)
random_forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", random_forest),
])

fit_start = time.perf_counter()
random_forest_pipeline.fit(X_train_sample, y_train_sample)
random_forest_training_time = time.perf_counter() - fit_start

validation_rows = 0
absolute_error_sum = 0.0
squared_error_sum = 0.0
target_sum = 0.0
target_squared_sum = 0.0
for row_group_index in range(validation_parquet.metadata.num_row_groups):
    table = validation_parquet.read_row_group(row_group_index, columns=MODEL_FEATURES + [TARGET])
    validation_frame = table.to_pandas()
    y_validation = validation_frame[TARGET].to_numpy(dtype="float64")
    predictions = random_forest_pipeline.predict(validation_frame[MODEL_FEATURES])
    errors = predictions - y_validation
    validation_rows += len(y_validation)
    absolute_error_sum += float(np.abs(errors).sum())
    squared_error_sum += float(np.square(errors).sum())
    target_sum += float(y_validation.sum())
    target_squared_sum += float(np.square(y_validation).sum())

validation_mean = target_sum / validation_rows
total_sum_of_squares = target_squared_sum - validation_rows * validation_mean**2
random_forest_mae = absolute_error_sum / validation_rows
random_forest_rmse = float(np.sqrt(squared_error_sum / validation_rows))
random_forest_r2 = 1 - squared_error_sum / total_sum_of_squares
assert validation_rows == EXPECTED_VALIDATION_ROWS

linear_regression_mae = 8.689478
linear_regression_rmse = 13.928211
linear_regression_r2 = 0.427485
baseline_mae = 11.713890
baseline_rmse = 19.724010
baseline_r2 = -0.148117
comparison = [
    ("Median baseline", "-", validation_rows, baseline_mae, baseline_rmse, baseline_r2, "-"),
    ("LinearRegression", 445_000, validation_rows, linear_regression_mae, linear_regression_rmse, linear_regression_r2, "7.003s"),
    ("RandomForestRegressor", len(training_sample), validation_rows, random_forest_mae, random_forest_rmse, random_forest_r2, f"{random_forest_training_time:.3f}s"),
]

print("Configuration: n_estimators=100, max_depth=18, min_samples_leaf=5, max_features=0.7, random_state=42, n_jobs=-1")
print("Training rows used:", len(training_sample))
print("Validation rows evaluated:", validation_rows)
print("Training time (seconds):", f"{random_forest_training_time:.3f}")
print("\n| Model | Train Rows | Validation Rows | MAE | RMSE | R² | Training Time |")
print("|---|---:|---:|---:|---:|---:|---:|")
for model, train_rows, valid_rows, mae, rmse, r2, training_time in comparison:
    print(f"| {model} | {train_rows if train_rows == '-' else f'{train_rows:,}'} | {valid_rows:,} | {mae:.6f} | {rmse:.6f} | {r2:.6f} | {training_time} |")
print("\nRandom Forest minus LinearRegression:")
print(f"MAE change: {random_forest_mae - linear_regression_mae:.6f}")
print(f"RMSE change: {random_forest_rmse - linear_regression_rmse:.6f}")
print(f"R² change: {random_forest_r2 - linear_regression_r2:.6f}")
print("Beats LinearRegression on all three metrics:", random_forest_mae < linear_regression_mae and random_forest_rmse < linear_regression_rmse and random_forest_r2 > linear_regression_r2)
print("PASS: RandomForestRegressor evaluated on validation only.")
print("WARNING: sampled training subset and fixed first-pass hyperparameters; no tuning performed.")
print("STOP: Step 6 RandomForestRegressor only. No test data or Gradient Boosting used.")

## Step 7: Additional nonlinear candidate

`HistGradientBoostingRegressor` is not used with the current Step 3 preprocessing because `OneHotEncoder` produces a sparse one-hot matrix, while HistGradientBoosting requires a dense numeric representation. Converting this large sparse matrix to dense would be memory-heavy and would change the established preprocessing setup.

As a compatible tree-based alternative, this step evaluates `ExtraTreesRegressor`, which accepts the current sparse one-hot representation inside the same preprocessing `Pipeline`. It uses the identical deterministic 445,000-row training sample as the prior models and fixed first-pass settings only; no tuning or test evaluation is performed.

### Interpretation

ExtraTreesRegressor does not beat LinearRegression on this first pass: MAE is 9.106238 versus 8.689478, RMSE is 14.121436 versus 13.928211, and R² is 0.411490 versus 0.427485. It does beat the median baseline and is slightly better than RandomForestRegressor on MAE, RMSE, and R², but it takes substantially longer to train than LinearRegression. The result does not justify replacing LinearRegression yet; another nonlinear candidate may be tested later, but only with fixed leakage-safe evaluation and without tuning in this step.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from scipy import sparse

print("## Step 7 additional nonlinear candidate verification")

MODEL_FEATURES = numeric_features + categorical_features
TARGET = "base_fare"
TRAIN_PATH = ROOT / "data" / "splits" / "train.parquet"
VALIDATION_PATH = ROOT / "data" / "splits" / "validation.parquet"
SAMPLE_PER_ROW_GROUP = 2_500
EXPECTED_TRAIN_SAMPLE_ROWS = 445_000
EXPECTED_VALIDATION_ROWS = 6_814_901

train_parquet = pq.ParquetFile(TRAIN_PATH)
validation_parquet = pq.ParquetFile(VALIDATION_PATH)
assert set(MODEL_FEATURES + [TARGET]).issubset(train_parquet.schema_arrow.names)
assert set(MODEL_FEATURES + [TARGET]).issubset(validation_parquet.schema_arrow.names)

training_sample_parts = []
for row_group_index in range(train_parquet.metadata.num_row_groups):
    table = train_parquet.read_row_group(row_group_index, columns=MODEL_FEATURES + [TARGET])
    row_count = table.num_rows
    sample_size = min(SAMPLE_PER_ROW_GROUP, row_count)
    indices = np.linspace(0, row_count - 1, num=sample_size, dtype=np.int64)
    training_sample_parts.append(table.take(pa.array(indices)).to_pandas())
training_sample = pd.concat(training_sample_parts, ignore_index=True)
X_train_sample = training_sample[MODEL_FEATURES]
y_train_sample = training_sample[TARGET].to_numpy(dtype="float64")
assert len(training_sample) == EXPECTED_TRAIN_SAMPLE_ROWS

extra_trees_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=100,
        max_depth=18,
        min_samples_leaf=5,
        max_features=0.7,
        random_state=42,
        n_jobs=-1,
    )),
])

fit_start = time.perf_counter()
extra_trees_pipeline.fit(X_train_sample, y_train_sample)
extra_trees_training_time = time.perf_counter() - fit_start
transformed_probe = extra_trees_pipeline.named_steps["preprocessor"].transform(X_train_sample.iloc[:1])
print("HistGradientBoosting compatibility with current sparse preprocessing: not used")
print("Current transformed representation is sparse:", sparse.issparse(transformed_probe))

validation_rows = 0
absolute_error_sum = 0.0
squared_error_sum = 0.0
target_sum = 0.0
target_squared_sum = 0.0
for row_group_index in range(validation_parquet.metadata.num_row_groups):
    table = validation_parquet.read_row_group(row_group_index, columns=MODEL_FEATURES + [TARGET])
    validation_frame = table.to_pandas()
    y_validation = validation_frame[TARGET].to_numpy(dtype="float64")
    predictions = extra_trees_pipeline.predict(validation_frame[MODEL_FEATURES])
    errors = predictions - y_validation
    validation_rows += len(y_validation)
    absolute_error_sum += float(np.abs(errors).sum())
    squared_error_sum += float(np.square(errors).sum())
    target_sum += float(y_validation.sum())
    target_squared_sum += float(np.square(y_validation).sum())

validation_mean = target_sum / validation_rows
total_sum_of_squares = target_squared_sum - validation_rows * validation_mean**2
extra_trees_mae = absolute_error_sum / validation_rows
extra_trees_rmse = float(np.sqrt(squared_error_sum / validation_rows))
extra_trees_r2 = 1 - squared_error_sum / total_sum_of_squares
assert validation_rows == EXPECTED_VALIDATION_ROWS

baseline_mae, baseline_rmse, baseline_r2 = 11.713890, 19.724010, -0.148117
linear_regression_mae, linear_regression_rmse, linear_regression_r2 = 8.689478, 13.928211, 0.427485
random_forest_mae, random_forest_rmse, random_forest_r2 = 9.209341, 14.128331, 0.410915
comparison = [
    ("Median baseline", "-", validation_rows, baseline_mae, baseline_rmse, baseline_r2, "-"),
    ("LinearRegression", 445_000, validation_rows, linear_regression_mae, linear_regression_rmse, linear_regression_r2, "7.003s"),
    ("RandomForestRegressor", 445_000, validation_rows, random_forest_mae, random_forest_rmse, random_forest_r2, "128.838s"),
    ("ExtraTreesRegressor", len(training_sample), validation_rows, extra_trees_mae, extra_trees_rmse, extra_trees_r2, f"{extra_trees_training_time:.3f}s"),
]

print("Configuration: n_estimators=100, max_depth=18, min_samples_leaf=5, max_features=0.7, random_state=42, n_jobs=-1")
print("Training rows used:", len(training_sample))
print("Validation rows evaluated:", validation_rows)
print("Training time (seconds):", f"{extra_trees_training_time:.3f}")
print("\n| Model | Train Rows | Validation Rows | MAE | RMSE | R² | Training Time |")
print("|---|---:|---:|---:|---:|---:|---:|")
for model, train_rows, valid_rows, mae, rmse, r2, training_time in comparison:
    print(f"| {model} | {train_rows if train_rows == '-' else f'{train_rows:,}'} | {valid_rows:,} | {mae:.6f} | {rmse:.6f} | {r2:.6f} | {training_time} |")
print("\nExtraTrees minus LinearRegression:")
print(f"MAE change: {extra_trees_mae - linear_regression_mae:.6f}")
print(f"RMSE change: {extra_trees_rmse - linear_regression_rmse:.6f}")
print(f"R² change: {extra_trees_r2 - linear_regression_r2:.6f}")
print("Beats LinearRegression on all three metrics:", extra_trees_mae < linear_regression_mae and extra_trees_rmse < linear_regression_rmse and extra_trees_r2 > linear_regression_r2)
print("PASS: ExtraTreesRegressor evaluated on validation only.")
print("WARNING: fixed sampled training subset and first-pass settings; no tuning performed.")
print("STOP: Step 7 additional nonlinear candidate only. No test data or HistGradientBoosting used.")

## Step 8: Controlled route_id feature experiment

This experiment adds only `route_id` to the finalized seven-feature LinearRegression setup. It recreates the same deterministic 445,000-row training sample and evaluates the same validation split. All other excluded features remain excluded, and the test split is not read.

The route-inclusive preprocessor is fitted on the training sample first so its encoded width can be checked. The route-inclusive representation produced **22,392 encoded features**, exceeding the safety limit of 5,000. The experiment therefore stopped before LinearRegression fitting rather than forcing an impractical high-cardinality computation. No route-inclusive MAE, RMSE, R², or improvement claim was produced.

**Decision:** route_id is not retained in this experiment because its one-hot representation is too high-cardinality for the current first-pass setup. A future route treatment would need a different encoding strategy approved separately; no such strategy is introduced here.

In [ ]:
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

print("## Step 8 route_id feature experiment")

ROUTE_NUMERIC_FEATURES = ["pickup_hour", "month"]
ROUTE_CATEGORICAL_FEATURES = [
    "provider_code",
    "day_of_week",
    "weekend",
    "origin_loc_id",
    "dest_loc_id",
    "route_id",
]
ROUTE_MODEL_FEATURES = ROUTE_NUMERIC_FEATURES + ROUTE_CATEGORICAL_FEATURES
TARGET = "base_fare"
TRAIN_PATH = ROOT / "data" / "splits" / "train.parquet"
VALIDATION_PATH = ROOT / "data" / "splits" / "validation.parquet"
SAMPLE_PER_ROW_GROUP = 2_500
EXPECTED_TRAIN_SAMPLE_ROWS = 445_000
EXPECTED_VALIDATION_ROWS = 6_814_901
MAX_ENCODED_FEATURES = 5_000

train_parquet = pq.ParquetFile(TRAIN_PATH)
validation_parquet = pq.ParquetFile(VALIDATION_PATH)
assert set(ROUTE_MODEL_FEATURES + [TARGET]).issubset(train_parquet.schema_arrow.names)
assert set(ROUTE_MODEL_FEATURES + [TARGET]).issubset(validation_parquet.schema_arrow.names)

training_sample_parts = []
for row_group_index in range(train_parquet.metadata.num_row_groups):
    table = train_parquet.read_row_group(row_group_index, columns=ROUTE_MODEL_FEATURES + [TARGET])
    row_count = table.num_rows
    sample_size = min(SAMPLE_PER_ROW_GROUP, row_count)
    indices = np.linspace(0, row_count - 1, num=sample_size, dtype=np.int64)
    training_sample_parts.append(table.take(pa.array(indices)).to_pandas())
training_sample = pd.concat(training_sample_parts, ignore_index=True)
assert len(training_sample) == EXPECTED_TRAIN_SAMPLE_ROWS

route_numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
route_categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
route_preprocessor = ColumnTransformer([
    ("numeric", route_numeric_preprocessor, ROUTE_NUMERIC_FEATURES),
    ("categorical", route_categorical_preprocessor, ROUTE_CATEGORICAL_FEATURES),
])

X_train_route = training_sample[ROUTE_MODEL_FEATURES]
y_train_route = training_sample[TARGET].to_numpy(dtype="float64")
route_preprocessor.fit(X_train_route, y_train_route)
encoded_feature_count = route_preprocessor.transform(X_train_route.iloc[:1]).shape[1]
print("Route-inclusive encoded feature count:", encoded_feature_count)
if encoded_feature_count > MAX_ENCODED_FEATURES:
    print(f"WARNING: encoded feature count exceeds safety limit ({MAX_ENCODED_FEATURES}); experiment stopped before model fitting.")
    print("No route-inclusive metrics were produced.")
else:
    route_pipeline = Pipeline([
        ("preprocessor", route_preprocessor),
        ("model", LinearRegression()),
    ])
    fit_start = time.perf_counter()
    route_pipeline.fit(X_train_route, y_train_route)
    route_training_time = time.perf_counter() - fit_start

    validation_rows = 0
    absolute_error_sum = 0.0
    squared_error_sum = 0.0
    target_sum = 0.0
    target_squared_sum = 0.0
    for row_group_index in range(validation_parquet.metadata.num_row_groups):
        table = validation_parquet.read_row_group(row_group_index, columns=ROUTE_MODEL_FEATURES + [TARGET])
        validation_frame = table.to_pandas()
        y_validation = validation_frame[TARGET].to_numpy(dtype="float64")
        predictions = route_pipeline.predict(validation_frame[ROUTE_MODEL_FEATURES])
        errors = predictions - y_validation
        validation_rows += len(y_validation)
        absolute_error_sum += float(np.abs(errors).sum())
        squared_error_sum += float(np.square(errors).sum())
        target_sum += float(y_validation.sum())
        target_squared_sum += float(np.square(y_validation).sum())

    validation_mean = target_sum / validation_rows
    total_sum_of_squares = target_squared_sum - validation_rows * validation_mean**2
    route_mae = absolute_error_sum / validation_rows
    route_rmse = float(np.sqrt(squared_error_sum / validation_rows))
    route_r2 = 1 - squared_error_sum / total_sum_of_squares
    assert validation_rows == EXPECTED_VALIDATION_ROWS

    original_mae = 8.689478
    original_rmse = 13.928211
    original_r2 = 0.427485
    original_training_time = 7.003
    mae_improvement = original_mae - route_mae
    rmse_improvement = original_rmse - route_rmse
    mae_improvement_percent = mae_improvement / original_mae * 100
    rmse_improvement_percent = rmse_improvement / original_rmse * 100

    print("Training rows used:", len(training_sample))
    print("Validation rows evaluated:", validation_rows)
    print("Training time (seconds):", f"{route_training_time:.3f}")
    print("\n| Feature Set | MAE | RMSE | R² | Training Time |")
    print("|---|---:|---:|---:|---:|")
    print(f"| Original 7 features | {original_mae:.6f} | {original_rmse:.6f} | {original_r2:.6f} | {original_training_time:.3f}s |")
    print(f"| Original 7 features + route_id | {route_mae:.6f} | {route_rmse:.6f} | {route_r2:.6f} | {route_training_time:.3f}s |")
    print("\nRoute_id improvement relative to original LinearRegression:")
    print(f"MAE improvement: {mae_improvement:.6f} ({mae_improvement_percent:.3f}%)")
    print(f"RMSE improvement: {rmse_improvement:.6f} ({rmse_improvement_percent:.3f}%)")
    print(f"R² change: {route_r2 - original_r2:.6f}")
    meaningful = mae_improvement_percent >= 1.0 and rmse_improvement_percent >= 1.0 and route_r2 > original_r2
    print("Meaningful enough to keep route_id:", meaningful)
    print("PASS: route_id experiment evaluated on validation only.")
    print("WARNING: same deterministic sampled training subset; no tuning or test evaluation performed.")
    print("STOP: Step 8 route_id experiment only.")

## Step 9: Conditional distance feature experiment

**This experiment assumes an estimated route distance is available at prediction time before the trip begins.**

`data/feature_contract.md` permits `distance_miles` conditionally under that assumption. The experiment uses the stored distance column as the conditional input; it does not establish that the stored values are actual pre-trip estimates or that an operational routing estimate would achieve these metrics. Results must not imply measured completed-trip distance is always available before pickup. Deployment would require an independently available pre-trip estimate and evaluation using that estimate.

Add **only `distance_miles`** to the winning seven-feature LinearRegression setup. Numeric inputs: `pickup_hour`, `month`, `distance_miles`. Categorical inputs: `provider_code`, `day_of_week`, `weekend`, `origin_loc_id`, `dest_loc_id`. No route ID, dates, zone labels, rider/rate/payment/offline fields, targets, or completed-trip outcomes enter the predictor.

Recreate the exact deterministic sample selection from Step 5: 2,500 evenly spaced positions per training row group, using `np.linspace(..., dtype=np.int64)`. Fit the unchanged median-imputer / most-frequent-imputer / one-hot preprocessing and default LinearRegression. A fresh seven-feature control is refit on those same rows to check reproducibility; this is not tuning. Evaluate both on all validation rows incrementally. Do not open test data or execute earlier notebook cells.

For a practical comparison, call an improvement meaningful if **both MAE and RMSE decrease by at least 1% and R2 increases**, matching the criterion used in Step 8. This is a practical decision criterion, not a statistical significance claim.

In [1]:
from pathlib import Path
import time,hashlib,json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import sklearn

d_root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"data"/"feature_contract.md").is_file())
d_contract_path=d_root/"data"/"feature_contract.md"
d_contract=d_contract_path.read_text(encoding="utf-8")
d_conditional=d_contract.split("### Conditional features",1)[1].split("## C.",1)[0]
assert "`distance_miles`" in d_conditional and "before departure" in d_conditional
print("Conditional contract permission:",next(line for line in d_conditional.splitlines() if "`distance_miles`" in line))
print("This experiment assumes an estimated route distance is available at prediction time before the trip begins.")
d_numeric=["pickup_hour","month"]
d_categorical=["provider_code","day_of_week","weekend","origin_loc_id","dest_loc_id"]
d_base_features=d_numeric+d_categorical
d_numeric_distance=d_numeric+["distance_miles"]
d_features=d_numeric_distance+d_categorical
d_target="base_fare"
assert set(d_features)-set(d_base_features)=={"distance_miles"} and len(d_features)==8
d_train_path=d_root/"data"/"splits"/"train.parquet"
d_validation_path=d_root/"data"/"splits"/"validation.parquet"
d_snapshot={str(p):(p.stat().st_size,p.stat().st_mtime_ns) for p in [d_train_path,d_validation_path,d_contract_path]}
d_train=pq.ParquetFile(d_train_path);d_validation=pq.ParquetFile(d_validation_path)
assert d_train.metadata.num_rows==31_988_176 and d_validation.metadata.num_rows==6_814_901
assert set(d_features+[d_target]).issubset(d_train.schema_arrow.names)
assert set(d_features+[d_target]).issubset(d_validation.schema_arrow.names)
d_parts=[];d_selection_digest=hashlib.sha256()
for group in range(d_train.metadata.num_row_groups):
    table=d_train.read_row_group(group,columns=d_features+[d_target])
    indices=np.linspace(0,table.num_rows-1,num=min(2_500,table.num_rows),dtype=np.int64)
    d_selection_digest.update(np.array([group,table.num_rows],dtype="int64").tobytes())
    d_selection_digest.update(indices.tobytes())
    d_parts.append(table.take(pa.array(indices)).to_pandas())
d_sample=pd.concat(d_parts,ignore_index=True)
d_parts.clear()
assert len(d_sample)==445_000
assert np.isfinite(d_sample[d_target].to_numpy()).all()
print("Training sample rows:",len(d_sample),"Training row groups:",d_train.metadata.num_row_groups)
print("Deterministic selection SHA-256:",d_selection_digest.hexdigest())
print("Distance dtype:",d_sample.distance_miles.dtype)
print("Training feature missingness:",d_sample[d_features].isna().sum().to_dict())
print("scikit-learn:",sklearn.__version__)


Conditional contract permission: | `distance_miles` | `double`; numeric | **Assumption required:** this is an upfront route estimate or requested distance available before departure, not GPS/measured distance calculated after completion. If it is only recorded actual distance, forbid it. | Fare and duration |
This experiment assumes an estimated route distance is available at prediction time before the trip begins.
Training sample rows: 445000 Training row groups: 178
Deterministic selection SHA-256: 903345475443eb13fa8f887fd00adaf31366e2185ab509808317dc47aa09d531
Distance dtype: float64
Training feature missingness: {'pickup_hour': 0, 'month': 0, 'distance_miles': 0, 'provider_code': 0, 'day_of_week': 0, 'weekend': 0, 'origin_loc_id': 0, 'dest_loc_id': 0}
scikit-learn: 1.9.0


### Fit the control and conditional-distance pipelines

Use fresh preprocessing objects for each model. Fit all imputation and encoding only on the identical training rows; validation targets never enter preprocessing. Training time includes `Pipeline.fit` (preprocessing plus estimator) and excludes sampling/evaluation. No scaling, clipping, hyperparameter changes, or additional features are introduced.

In [2]:
def d_make_pipeline(numeric):
    preprocessor=ColumnTransformer([
        ("numeric",Pipeline([("imputer",SimpleImputer(strategy="median"))]),numeric),
        ("categorical",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),d_categorical),
    ])
    return Pipeline([("preprocessor",preprocessor),("model",LinearRegression())])

d_models={"Original 7 features":d_make_pipeline(d_numeric),"7 features + distance (conditional)":d_make_pipeline(d_numeric_distance)}
d_model_features={"Original 7 features":d_base_features,"7 features + distance (conditional)":d_features}
d_training_times={}
for name,model in d_models.items():
    print("Fitting:",name,flush=True)
    begin=time.perf_counter()
    model.fit(d_sample[d_model_features[name]],d_sample[d_target].to_numpy(dtype="float64"))
    d_training_times[name]=time.perf_counter()-begin
    print(f"Finished {name}: {d_training_times[name]:.3f} seconds",flush=True)
    print("Encoded columns:",model.named_steps["preprocessor"].transform(d_sample[d_model_features[name]].iloc[:1]).shape[1])


Fitting: Original 7 features
Finished Original 7 features: 9.282 seconds
Encoded columns: 526
Fitting: 7 features + distance (conditional)
Finished 7 features + distance (conditional): 7.346 seconds
Encoded columns: 527


### Full validation and comparison

Both models predict exactly the same full validation split. Accumulate absolute and squared errors, and the target sums needed for R2, without storing all predictions. Positive improvement means lower error. Compare against the previously recorded winning results and report the new control run separately; historical training time is not a timing benchmark for this machine state.

In [3]:
d_acc={name:{"absolute_error":0.,"squared_error":0.} for name in d_models}
d_n=0;d_y_sum=0.;d_y_sq_sum=0.
for group in range(d_validation.metadata.num_row_groups):
    frame=d_validation.read_row_group(group,columns=d_features+[d_target]).to_pandas()
    y=frame[d_target].to_numpy(dtype="float64")
    assert np.isfinite(y).all()
    for name,model in d_models.items():
        prediction=model.predict(frame[d_model_features[name]])
        assert np.isfinite(prediction).all()
        error=prediction-y
        d_acc[name]["absolute_error"]+=float(np.abs(error).sum())
        d_acc[name]["squared_error"]+=float(np.square(error).sum())
    d_n+=len(y);d_y_sum+=float(y.sum());d_y_sq_sum+=float(np.square(y).sum())
assert d_n==6_814_901
sst=d_y_sq_sum-d_y_sum*d_y_sum/d_n
assert sst>0
d_results={name:{"MAE":a["absolute_error"]/d_n,"RMSE":float(np.sqrt(a["squared_error"]/d_n)),"R2":1-a["squared_error"]/sst,"Training Time":d_training_times[name]} for name,a in d_acc.items()}
d_previous={"MAE":8.689478,"RMSE":13.928211,"R2":0.427485,"Training Time":7.003}
d_control=d_results["Original 7 features"];d_experiment=d_results["7 features + distance (conditional)"]
d_reproduced=all(abs(d_control[key]-d_previous[key])<=1e-5 for key in ["MAE","RMSE","R2"])
print("Full validation rows:",d_n)
print("Control reproduces recorded baseline within 1e-5:",d_reproduced)
print("| Feature Set | MAE | RMSE | R2 | Training Time |")
print("|---|---:|---:|---:|---:|")
for name,metrics in [("Original 7 features (recorded)",d_previous),("Original 7 features (control rerun)",d_control),("7 features + distance (conditional)",d_experiment)]:
    print(f"| {name} | {metrics['MAE']:.6f} | {metrics['RMSE']:.6f} | {metrics['R2']:.6f} | {metrics['Training Time']:.3f}s |")
d_improvement={}
for key in ["MAE","RMSE"]:
    absolute=d_previous[key]-d_experiment[key]
    d_improvement[key]={"absolute":absolute,"percent":100*absolute/d_previous[key]}
    print(f"{key} improvement versus recorded baseline: {absolute:.6f} ({d_improvement[key]['percent']:.3f}%)")
d_meaningful=d_improvement['MAE']['percent']>=1 and d_improvement['RMSE']['percent']>=1 and d_experiment['R2']>d_previous['R2']
print("Meaningful under stated practical criterion:",d_meaningful)
if not d_reproduced:print("WARNING: control differs from recorded baseline; use the paired rerun for causal feature comparison.")
print("Paired-control MAE improvement:",d_control['MAE']-d_experiment['MAE'])
print("Paired-control RMSE improvement:",d_control['RMSE']-d_experiment['RMSE'])
assert d_snapshot=={str(p):(p.stat().st_size,p.stat().st_mtime_ns) for p in [d_train_path,d_validation_path,d_contract_path]}
d_train.close();d_validation.close()
print("RESULT_JSON="+json.dumps({"training_rows":len(d_sample),"validation_rows":d_n,"results":d_results,"recorded_baseline":d_previous,"improvement":d_improvement,"control_reproduced":d_reproduced,"meaningful":bool(d_meaningful)}))
print("STOP: Step 9 conditional-feature experiment only. No test data opened, tuning, duration prediction, or Member 1 data edits.")


Full validation rows: 6814901
Control reproduces recorded baseline within 1e-5: True
| Feature Set | MAE | RMSE | R2 | Training Time |
|---|---:|---:|---:|---:|
| Original 7 features (recorded) | 8.689478 | 13.928211 | 0.427485 | 7.003s |
| Original 7 features (control rerun) | 8.689478 | 13.928211 | 0.427485 | 9.282s |
| 7 features + distance (conditional) | 5.737909 | 10.986343 | 0.643793 | 7.346s |
MAE improvement versus recorded baseline: 2.951569 (33.967%)
RMSE improvement versus recorded baseline: 2.941868 (21.122%)
Meaningful under stated practical criterion: True
Paired-control MAE improvement: 2.951568320260483
Paired-control RMSE improvement: 2.9418680683866274
RESULT_JSON={"training_rows": 445000, "validation_rows": 6814901, "results": {"Original 7 features": {"MAE": 8.689477569244872, "RMSE": 13.928211488148033, "R2": 0.4274853343885754, "Training Time": 9.282282599946484}, "7 features + distance (conditional)": {"MAE": 5.737909248984389, "RMSE": 10.986343419761406, "R2": 0

### Step 9 result: meaningful improvement under the conditional-feature assumption

**This experiment assumes an estimated route distance is available at prediction time before the trip begins.**

Both pipelines used the identical deterministic **445,000 training rows** and all **6,814,901 validation rows**. The fresh seven-feature control reproduced the historical winning metrics within 0.00001. Only `distance_miles` was added; preprocessing and default LinearRegression settings were unchanged.

| Feature Set | MAE | RMSE | R2 | Training Time |
|---|---:|---:|---:|---:|
| Original 7 features (control rerun) | 8.689478 | 13.928211 | 0.427485 | 9.282s |
| Original 7 features + distance (conditional) | 5.737909 | 10.986343 | 0.643793 | 7.346s |

Relative to the recorded baseline (MAE 8.689478, RMSE 13.928211):

- **MAE improvement:** 2.951569, or **33.967%**.
- **RMSE improvement:** 2.941868, or **21.122%**.
- **R2 increase:** 0.216308.

This comfortably meets the prespecified practical criterion (at least 1% improvement in both errors and higher R2). **Distance is a meaningful addition in this conditional experiment.** It does not replace the unconditional seven-feature model for settings where no pre-trip route estimate exists. The stored distance was used under the stated assumption; its actual serving-time provenance is not established by this experiment, and performance with genuine route estimates must be checked before operational use. No statistical-significance or repeated-run timing claim is made. The previously recorded baseline fit time was 7.003 seconds; the table reports this run's timings.

Earlier fare notebook cells are preserved. Train/validation inputs and the feature contract were unchanged during this experiment. No test data was opened, hyperparameters tuned, models persisted, Member 1 data modified, or duration prediction started. **Stop after Step 9.**

## Step 10: Final fare model selection and validation error analysis

This step finalizes the model choice from validation evidence only. The strict pre-trip configuration remains the seven-feature `LinearRegression` without distance. The preferred route-estimate configuration adds only `distance_miles` and is valid only under this explicit assumption:

> **An estimated route distance is available at prediction time before the trip begins.**

The stored `distance_miles` field is used as the experimental proxy. This does not prove that it is a pre-trip estimate or that a production routing estimate will reproduce these results. If that assumption cannot be enforced, use the strict seven-feature configuration.

The selected route-estimate pipeline is refit on the same deterministic 445,000-row training sample and evaluated on the same full validation split. No test data is opened, no hyperparameters are tuned, and no model artifact is saved.

In [4]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

s10_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "splits").is_dir())
s10_train_path = s10_root / "data" / "splits" / "train.parquet"
s10_validation_path = s10_root / "data" / "splits" / "validation.parquet"
s10_test_path = s10_root / "data" / "splits" / "test.parquet"
s10_figure_dir = s10_root / "reports" / "figures"
s10_figure_dir.mkdir(parents=True, exist_ok=True)

s10_numeric = ["pickup_hour", "month", "distance_miles"]
s10_categorical = ["provider_code", "day_of_week", "weekend", "origin_loc_id", "dest_loc_id"]
s10_features = s10_numeric + s10_categorical
s10_target = "base_fare"
s10_train = pq.ParquetFile(s10_train_path)
s10_validation = pq.ParquetFile(s10_validation_path)
assert s10_train.metadata.num_rows == 31_988_176
assert s10_validation.metadata.num_rows == 6_814_901
assert set(s10_features + [s10_target]).issubset(s10_train.schema_arrow.names)
assert set(s10_features + [s10_target]).issubset(s10_validation.schema_arrow.names)

# Recreate the established deterministic sample: 2,500 evenly spaced rows per row group.
s10_parts = []
for row_group_index in range(s10_train.metadata.num_row_groups):
    table = s10_train.read_row_group(row_group_index, columns=s10_features + [s10_target])
    indices = np.linspace(0, table.num_rows - 1, num=min(2_500, table.num_rows), dtype=np.int64)
    s10_parts.append(table.take(pa.array(indices)).to_pandas())
s10_sample = pd.concat(s10_parts, ignore_index=True)
del s10_parts
assert len(s10_sample) == 445_000

s10_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), s10_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), s10_categorical),
])
s10_pipeline = Pipeline([
    ("preprocessor", s10_preprocessor),
    ("model", LinearRegression()),
])
s10_fit_start = time.perf_counter()
s10_pipeline.fit(s10_sample[s10_features], s10_sample[s10_target].to_numpy(dtype="float64"))
s10_training_time = time.perf_counter() - s10_fit_start
print("Selected model: LinearRegression with conditional distance_miles")
print("Training rows:", len(s10_sample))
print(f"Training time: {s10_training_time:.3f} seconds")
print("No test data opened; test path recorded only for post-run filesystem verification:", s10_test_path.name)


Selected model: LinearRegression with conditional distance_miles
Training rows: 445000
Training time: 7.727 seconds
No test data opened; test path recorded only for post-run filesystem verification: test.parquet


### Full validation error analysis

Stream every validation row group and accumulate metrics and subgroup errors without retaining all predictions. MAE by pickup hour and distance bucket is exact over the full validation split. Distance buckets are left-closed: 0–2 means `[0, 2)`, 2–5 means `[2, 5)`, 5–10 means `[5, 10)`, 10–20 means `[10, 20)`, and 20+ means `[20, infinity)` miles.

The predicted-versus-actual and residual plots use a deterministic, evenly spaced plotting sample from every validation row group so millions of overlapping points remain legible. Overall metrics and grouped MAEs still use all 6,814,901 validation rows.

In [5]:
s10_candidate_results = pd.DataFrame([
    {"Model": "Median baseline", "Configuration": "Constant training median", "MAE": 11.713890, "RMSE": 19.724010, "R2": -0.148117},
    {"Model": "LinearRegression", "Configuration": "Strict pre-trip; no distance", "MAE": 8.689478, "RMSE": 13.928211, "R2": 0.427485},
    {"Model": "RandomForestRegressor", "Configuration": "Strict pre-trip; no distance", "MAE": 9.209341, "RMSE": 14.128331, "R2": 0.410915},
    {"Model": "ExtraTreesRegressor", "Configuration": "Strict pre-trip; no distance", "MAE": 9.106238, "RMSE": 14.121436, "R2": 0.411490},
    {"Model": "LinearRegression", "Configuration": "Conditional pre-trip route estimate", "MAE": 5.737909, "RMSE": 10.986343, "R2": 0.643793},
])
print("Final candidate comparison (recorded validation results):")
print(s10_candidate_results.to_string(index=False))

s10_n = 0
s10_abs_sum = 0.0
s10_sq_sum = 0.0
s10_y_sum = 0.0
s10_y_sq_sum = 0.0
s10_hour_abs = np.zeros(24, dtype="float64")
s10_hour_count = np.zeros(24, dtype="int64")
s10_bucket_labels = ["0–2 miles", "2–5 miles", "5–10 miles", "10–20 miles", "20+ miles"]
s10_bucket_edges = [0, 2, 5, 10, 20, np.inf]
s10_bucket_abs = np.zeros(5, dtype="float64")
s10_bucket_count = np.zeros(5, dtype="int64")
s10_plot_actual = []
s10_plot_predicted = []

for row_group_index in range(s10_validation.metadata.num_row_groups):
    frame = s10_validation.read_row_group(row_group_index, columns=s10_features + [s10_target]).to_pandas()
    actual = frame[s10_target].to_numpy(dtype="float64")
    predicted = s10_pipeline.predict(frame[s10_features])
    residual = actual - predicted
    absolute = np.abs(residual)
    s10_n += len(actual)
    s10_abs_sum += float(absolute.sum())
    s10_sq_sum += float(np.square(residual).sum())
    s10_y_sum += float(actual.sum())
    s10_y_sq_sum += float(np.square(actual).sum())
    hours = frame["pickup_hour"].to_numpy(dtype="int64")
    s10_hour_abs += np.bincount(hours, weights=absolute, minlength=24)
    s10_hour_count += np.bincount(hours, minlength=24)
    buckets = pd.cut(frame["distance_miles"], bins=s10_bucket_edges, labels=False, right=False, include_lowest=True)
    assert buckets.notna().all(), "Distance outside documented nonnegative bucket range"
    bucket_codes = buckets.to_numpy(dtype="int64")
    s10_bucket_abs += np.bincount(bucket_codes, weights=absolute, minlength=5)
    s10_bucket_count += np.bincount(bucket_codes, minlength=5)
    plot_count = min(2_000, len(frame))
    plot_indices = np.linspace(0, len(frame) - 1, num=plot_count, dtype=np.int64)
    s10_plot_actual.append(actual[plot_indices])
    s10_plot_predicted.append(predicted[plot_indices])

s10_actual = np.concatenate(s10_plot_actual)
s10_predicted = np.concatenate(s10_plot_predicted)
s10_residual = s10_actual - s10_predicted
s10_sst = s10_y_sq_sum - s10_y_sum**2 / s10_n
s10_metrics = {
    "MAE": s10_abs_sum / s10_n,
    "RMSE": float(np.sqrt(s10_sq_sum / s10_n)),
    "R2": 1 - s10_sq_sum / s10_sst,
}
assert s10_n == 6_814_901
assert abs(s10_metrics["MAE"] - 5.737909) < 1e-5
assert abs(s10_metrics["RMSE"] - 10.986343) < 1e-5
assert abs(s10_metrics["R2"] - 0.643793) < 1e-5
s10_hour_table = pd.DataFrame({"pickup_hour": np.arange(24), "rows": s10_hour_count, "MAE": s10_hour_abs / s10_hour_count})
s10_distance_table = pd.DataFrame({"distance_bucket": s10_bucket_labels, "rows": s10_bucket_count, "MAE": s10_bucket_abs / s10_bucket_count})
assert s10_hour_count.sum() == s10_bucket_count.sum() == s10_n
print("Full validation metrics:", s10_metrics)
print("MAE by pickup hour:")
print(s10_hour_table.to_string(index=False))
print("MAE by distance bucket:")
print(s10_distance_table.to_string(index=False))


Final candidate comparison (recorded validation results):
                Model                       Configuration       MAE      RMSE        R2
      Median baseline            Constant training median 11.713890 19.724010 -0.148117
     LinearRegression        Strict pre-trip; no distance  8.689478 13.928211  0.427485
RandomForestRegressor        Strict pre-trip; no distance  9.209341 14.128331  0.410915
  ExtraTreesRegressor        Strict pre-trip; no distance  9.106238 14.121436  0.411490
     LinearRegression Conditional pre-trip route estimate  5.737909 10.986343  0.643793
Full validation metrics: {'MAE': 5.737909248984389, 'RMSE': 10.986343419761406, 'R2': 0.6437931623377606}
MAE by pickup hour:
 pickup_hour   rows      MAE
           0 214657 5.835211
           1 144344 5.679667
           2  97035 4.974693
           3  68053 4.947629
           4  54680 6.752896
           5  59860 6.461779
           6 114639 6.361459
           7 198985 6.698825
           8 262143 6.59389

### Report-ready figures

Each figure includes its interpretation inside the image. Predicted-versus-actual and residual plots are visualization samples only; exact validation metrics and group MAEs are printed above.

In [6]:
import json
s10_style = {"figure.facecolor": "white", "axes.facecolor": "#f7f9fc", "axes.edgecolor": "#364152", "grid.alpha": 0.25}
plt.rcParams.update(s10_style)

# Predicted versus actual: use robust shared limits so the main cloud is readable, while reporting the sampling scope.
s10_lo = float(min(np.quantile(s10_actual, 0.005), np.quantile(s10_predicted, 0.005)))
s10_hi = float(max(np.quantile(s10_actual, 0.995), np.quantile(s10_predicted, 0.995)))
fig, ax = plt.subplots(figsize=(9, 7))
plot = ax.hexbin(s10_actual, s10_predicted, gridsize=80, bins="log", mincnt=1, cmap="viridis")
ax.plot([s10_lo, s10_hi], [s10_lo, s10_hi], linestyle="--", color="#dc2626", linewidth=1.5, label="Perfect prediction")
ax.set_xlim(s10_lo, s10_hi); ax.set_ylim(s10_lo, s10_hi)
ax.set_title("Fare prediction: predicted vs actual base fare")
ax.set_xlabel("Actual base fare (fare units)"); ax.set_ylabel("Predicted base fare (fare units)")
ax.legend(); fig.colorbar(plot, ax=ax, label="log10 sampled observation count")
fig.text(0.5, 0.01, "Interpretation: predictions track the main fare trend, while dispersion grows among higher fares.", ha="center", fontsize=9)
fig.tight_layout(rect=[0, 0.04, 1, 1]); fig.savefig(s10_figure_dir / "fare_predicted_vs_actual.png", dpi=180); plt.close(fig)

# Residual distribution: central 99% shown for legibility and clearly labeled.
s10_residual_limit = float(np.quantile(np.abs(s10_residual), 0.995))
fig, ax = plt.subplots(figsize=(9, 6))
ax.hist(s10_residual, bins=120, range=(-s10_residual_limit, s10_residual_limit), color="#2563eb", alpha=0.85)
ax.axvline(0, color="#dc2626", linestyle="--", linewidth=1.5)
ax.set_title("Fare-model residual distribution (central 99% of plotting sample)")
ax.set_xlabel("Residual: actual − predicted base fare (fare units)"); ax.set_ylabel("Sampled validation rows (count)")
fig.text(0.5, 0.01, "Interpretation: residuals cluster near zero but retain a right tail from underpredicted high fares.", ha="center", fontsize=9)
fig.tight_layout(rect=[0, 0.04, 1, 1]); fig.savefig(s10_figure_dir / "fare_residual_distribution.png", dpi=180); plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(s10_hour_table["pickup_hour"], s10_hour_table["MAE"], color="#0f766e")
ax.set_title("Validation MAE by pickup hour")
ax.set_xlabel("Pickup hour (0–23)"); ax.set_ylabel("Mean absolute error (fare units)")
ax.set_xticks(np.arange(24)); ax.grid(axis="y")
s10_best_hour = int(s10_hour_table.loc[s10_hour_table.MAE.idxmin(), "pickup_hour"])
s10_worst_hour = int(s10_hour_table.loc[s10_hour_table.MAE.idxmax(), "pickup_hour"])
fig.text(0.5, 0.01, f"Interpretation: error is lowest at hour {s10_best_hour:02d}:00 and highest at hour {s10_worst_hour:02d}:00.", ha="center", fontsize=9)
fig.tight_layout(rect=[0, 0.04, 1, 1]); fig.savefig(s10_figure_dir / "fare_mae_by_hour.png", dpi=180); plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 6))
ax.bar(s10_distance_table["distance_bucket"], s10_distance_table["MAE"], color="#7c3aed")
ax.set_title("Validation MAE by estimated-distance bucket")
ax.set_xlabel("Distance bucket (miles)"); ax.set_ylabel("Mean absolute error (fare units)")
ax.grid(axis="y")
s10_best_bucket = s10_distance_table.loc[s10_distance_table.MAE.idxmin(), "distance_bucket"]
s10_worst_bucket = s10_distance_table.loc[s10_distance_table.MAE.idxmax(), "distance_bucket"]
fig.text(0.5, 0.01, f"Interpretation: error is lowest for {s10_best_bucket} and highest for {s10_worst_bucket}.", ha="center", fontsize=9)
fig.tight_layout(rect=[0, 0.04, 1, 1]); fig.savefig(s10_figure_dir / "fare_mae_by_distance_bucket.png", dpi=180); plt.close(fig)

s10_figure_paths = [
    s10_figure_dir / "fare_predicted_vs_actual.png",
    s10_figure_dir / "fare_residual_distribution.png",
    s10_figure_dir / "fare_mae_by_hour.png",
    s10_figure_dir / "fare_mae_by_distance_bucket.png",
]
for path in s10_figure_paths:
    assert path.exists() and path.stat().st_size > 10_000
    print(path.relative_to(s10_root), path.stat().st_size, "bytes")

s10_summary = {
    "selected_model": "LinearRegression",
    "configuration": "conditional pre-trip route estimate",
    "features": s10_features,
    "training_rows": len(s10_sample),
    "validation_rows": s10_n,
    "training_time_seconds": s10_training_time,
    **s10_metrics,
    "best_hour": s10_best_hour,
    "best_hour_mae": float(s10_hour_table.loc[s10_hour_table.pickup_hour.eq(s10_best_hour), "MAE"].iloc[0]),
    "worst_hour": s10_worst_hour,
    "worst_hour_mae": float(s10_hour_table.loc[s10_hour_table.pickup_hour.eq(s10_worst_hour), "MAE"].iloc[0]),
    "best_distance_bucket": s10_best_bucket,
    "best_distance_bucket_mae": float(s10_distance_table.loc[s10_distance_table.distance_bucket.eq(s10_best_bucket), "MAE"].iloc[0]),
    "worst_distance_bucket": s10_worst_bucket,
    "worst_distance_bucket_mae": float(s10_distance_table.loc[s10_distance_table.distance_bucket.eq(s10_worst_bucket), "MAE"].iloc[0]),
}
print("STEP10_JSON=" + json.dumps(s10_summary))
print("PASS: validation-only error analysis complete.")
print("STOP: Step 10 only. No test use, train+validation refit, model save, tuning, duration prediction, or Member 1 data changes.")


reports\figures\fare_predicted_vs_actual.png 230910 bytes
reports\figures\fare_residual_distribution.png 62805 bytes
reports\figures\fare_mae_by_hour.png 52581 bytes
reports\figures\fare_mae_by_distance_bucket.png 56124 bytes
STEP10_JSON={"selected_model": "LinearRegression", "configuration": "conditional pre-trip route estimate", "features": ["pickup_hour", "month", "distance_miles", "provider_code", "day_of_week", "weekend", "origin_loc_id", "dest_loc_id"], "training_rows": 445000, "validation_rows": 6814901, "training_time_seconds": 7.726839300012216, "MAE": 5.737909248984389, "RMSE": 10.986343419761406, "R2": 0.6437931623377606, "best_hour": 10, "best_hour_mae": 4.640804241479008, "worst_hour": 17, "worst_hour_mae": 7.030075604665572, "best_distance_bucket": "0\u20132 miles", "best_distance_bucket_mae": 5.225641361418747, "worst_distance_bucket": "20+ miles", "worst_distance_bucket_mae": 16.788648670454986}
PASS: validation-only error analysis complete.
STOP: Step 10 only. No test 

### Step 10 decision

| Configuration | Model and features | Validation result | Decision |
|---|---|---|---|
| Strict pre-trip | `LinearRegression`; `pickup_hour`, `month`, `provider_code`, `day_of_week`, `weekend`, `origin_loc_id`, `dest_loc_id` | MAE 8.689478; RMSE 13.928211; R2 0.427485 | Use when no reliable route estimate exists. |
| Route estimate | `LinearRegression`; strict features plus `distance_miles` | MAE 5.737909; RMSE 10.986343; R2 0.643793 | **Preferred only when a pre-trip estimated route distance is available and enforced.** |

The route-estimate configuration is the best defensible validation model under its availability assumption: it has the lowest MAE and RMSE and highest R2 among all evaluated candidates, while retaining the simpler and faster LinearRegression model family. It improves MAE by **33.967%** and RMSE by **21.122%** over strict LinearRegression. Random Forest and Extra Trees are not tuned because Step 10 compares the completed fixed candidates rather than opening another model-search loop.

The clearest error-analysis result is the distance gradient. MAE is **5.225641 fare units** for 0–2 miles and **16.788649** for 20+ miles, so long trips remain materially harder despite distance being present. By hour, MAE is lowest at **10:00 (4.640804)** and highest at **17:00 (7.030076)**. The predicted-versus-actual plot shows increasing dispersion at higher fares, while sampled residuals retain a right tail, indicating underprediction of some expensive trips.

The most important limitation is feature availability: this result uses stored `distance_miles` as a proxy under an explicit pre-trip-route-estimate assumption. If the stored field is completed-trip mileage, it is unavailable at serving time; a production route estimator may also introduce error and distribution shift. The strict seven-feature model therefore remains the defensible fallback. A second limitation is weaker performance for 20+ mile trips, which comprise 56,697 validation rows and have roughly 3.21 times the MAE of 0–2 mile trips.

**Status: PASS with a conditional-feature warning.** Validation-only selection and error analysis are complete. No test data was opened, no train-plus-validation refit or model serialization occurred, no hyperparameters were tuned, and Member 1 data was not modified during Step 10. Stop after Step 10.

## Step 11: Final test evaluation

Model selection is complete. No further feature changes, model selection, or hyperparameter tuning will occur. This step performs the one final test evaluation, reports it against the already fixed validation metrics, and makes no changes after observing the result.

Selected model: default `LinearRegression` inside the established preprocessing `Pipeline`. Numeric features are `pickup_hour`, `month`, and conditional `distance_miles`; categorical features are `provider_code`, `day_of_week`, `weekend`, `origin_loc_id`, and `dest_loc_id`. Target: `base_fare`.

> **The distance-inclusive model assumes an estimated route distance is available before the trip begins.** The stored distance is an experimental proxy; completed-trip mileage must not be presented as known before pickup.

The final fit uses the established deterministic sampling philosophy: take up to 2,500 evenly spaced rows from every row group in both train and validation. Validation can now contribute to fitting because model selection has ended. This is a sampled final fit, not a claim that all 38.8 million train-plus-validation rows were used.

The final test cell is saved with `STEP11_ALLOW_FINAL_TEST_EVALUATION = False`. Its recorded output was produced once under the user's authorization, after which the guard remained locked to prevent accidental reruns. Do not unlock it or evaluate test again during model development.

In [7]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# This guard is intentionally committed in the locked state after the single authorized run.
STEP11_ALLOW_FINAL_TEST_EVALUATION = False
if not STEP11_ALLOW_FINAL_TEST_EVALUATION:
    raise RuntimeError("Final test evaluation already completed. Do not rerun or unlock during model development.")

s11_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "splits").is_dir())
s11_train_path = s11_root / "data" / "splits" / "train.parquet"
s11_validation_path = s11_root / "data" / "splits" / "validation.parquet"
s11_test_path = s11_root / "data" / "splits" / "test.parquet"
s11_numeric = ["pickup_hour", "month", "distance_miles"]
s11_categorical = ["provider_code", "day_of_week", "weekend", "origin_loc_id", "dest_loc_id"]
s11_features = s11_numeric + s11_categorical
s11_target = "base_fare"

print("MODEL SELECTION COMPLETE: confirmed")
print("NO FURTHER TUNING: confirmed")
print("FINAL TEST WILL BE EVALUATED ONCE ONLY: confirmed")
print("Conditional assumption: estimated route distance is available before trip start.")

s11_parts = []
s11_sample_counts = {}
for split_name, split_path, expected_rows in [
    ("train", s11_train_path, 31_988_176),
    ("validation", s11_validation_path, 6_814_901),
]:
    parquet = pq.ParquetFile(split_path)
    assert parquet.metadata.num_rows == expected_rows
    assert set(s11_features + [s11_target]).issubset(parquet.schema_arrow.names)
    sampled_rows = 0
    for row_group_index in range(parquet.metadata.num_row_groups):
        table = parquet.read_row_group(row_group_index, columns=s11_features + [s11_target])
        sample_size = min(2_500, table.num_rows)
        indices = np.linspace(0, table.num_rows - 1, num=sample_size, dtype=np.int64)
        s11_parts.append(table.take(pa.array(indices)).to_pandas())
        sampled_rows += sample_size
    parquet.close()
    s11_sample_counts[split_name] = sampled_rows

s11_training_sample = pd.concat(s11_parts, ignore_index=True)
del s11_parts
assert s11_sample_counts == {"train": 445_000, "validation": 97_500}
assert len(s11_training_sample) == 542_500

s11_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), s11_numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), s11_categorical),
])
s11_pipeline = Pipeline([
    ("preprocessor", s11_preprocessor),
    ("model", LinearRegression()),
])

s11_fit_start = time.perf_counter()
s11_pipeline.fit(
    s11_training_sample[s11_features],
    s11_training_sample[s11_target].to_numpy(dtype="float64"),
)
s11_training_time = time.perf_counter() - s11_fit_start
print("Train rows used in final fit:", s11_sample_counts["train"])
print("Validation rows used in final fit:", s11_sample_counts["validation"])
print("Total final-training rows:", len(s11_training_sample))
print(f"Final training time: {s11_training_time:.3f} seconds")

# Single test access and single full streaming evaluation pass.
s11_test_open_count = 0
s11_test_stream_passes = 0
s11_test_open_count += 1
s11_test = pq.ParquetFile(s11_test_path)
assert s11_test.metadata.num_rows == 6_730_257
assert set(s11_features + [s11_target]).issubset(s11_test.schema_arrow.names)
s11_test_stream_passes += 1
s11_n = 0
s11_abs_sum = 0.0
s11_sq_sum = 0.0
s11_y_sum = 0.0
s11_y_sq_sum = 0.0
for row_group_index in range(s11_test.metadata.num_row_groups):
    frame = s11_test.read_row_group(row_group_index, columns=s11_features + [s11_target]).to_pandas()
    actual = frame[s11_target].to_numpy(dtype="float64")
    prediction = s11_pipeline.predict(frame[s11_features])
    error = prediction - actual
    s11_n += len(actual)
    s11_abs_sum += float(np.abs(error).sum())
    s11_sq_sum += float(np.square(error).sum())
    s11_y_sum += float(actual.sum())
    s11_y_sq_sum += float(np.square(actual).sum())
s11_test.close()
assert s11_test_open_count == 1 and s11_test_stream_passes == 1
assert s11_n == 6_730_257
s11_sst = s11_y_sq_sum - s11_y_sum**2 / s11_n
s11_test_metrics = {
    "MAE": s11_abs_sum / s11_n,
    "RMSE": float(np.sqrt(s11_sq_sum / s11_n)),
    "R2": 1 - s11_sq_sum / s11_sst,
}
s11_validation_metrics = {"MAE": 5.737909, "RMSE": 10.986343, "R2": 0.643793}
s11_difference = {metric: s11_test_metrics[metric] - s11_validation_metrics[metric] for metric in s11_test_metrics}

print("| Dataset | MAE | RMSE | R2 | Rows |")
print("|---|---:|---:|---:|---:|")
print(f"| Validation | {s11_validation_metrics['MAE']:.6f} | {s11_validation_metrics['RMSE']:.6f} | {s11_validation_metrics['R2']:.6f} | 6,814,901 |")
print(f"| Final Test | {s11_test_metrics['MAE']:.6f} | {s11_test_metrics['RMSE']:.6f} | {s11_test_metrics['R2']:.6f} | {s11_n:,} |")
print("Test minus validation differences:")
for metric, difference in s11_difference.items():
    print(f"{metric}: {difference:+.6f}")

# A predeclared reporting heuristic; it does not trigger model changes.
s11_relative_mae_change = s11_difference["MAE"] / s11_validation_metrics["MAE"] * 100
s11_relative_rmse_change = s11_difference["RMSE"] / s11_validation_metrics["RMSE"] * 100
s11_stable = (
    abs(s11_relative_mae_change) <= 10
    and abs(s11_relative_rmse_change) <= 10
    and abs(s11_difference["R2"]) <= 0.05
)
s11_result = {
    "train_sample_rows": s11_sample_counts["train"],
    "validation_sample_rows": s11_sample_counts["validation"],
    "total_training_rows": len(s11_training_sample),
    "training_time_seconds": s11_training_time,
    "test_rows": s11_n,
    "test_metrics": s11_test_metrics,
    "validation_metrics": s11_validation_metrics,
    "test_minus_validation": s11_difference,
    "relative_mae_change_percent": s11_relative_mae_change,
    "relative_rmse_change_percent": s11_relative_rmse_change,
    "stable_under_reporting_heuristic": bool(s11_stable),
    "test_open_count": s11_test_open_count,
    "test_stream_passes": s11_test_stream_passes,
}
print("STEP11_JSON=" + json.dumps(s11_result))
print("TEST USED ONCE ONLY: confirmed; one open and one streaming pass.")
print("NO MODEL OR FEATURE CHANGES AFTER TEST: confirmed.")
print("STOP: Step 11 final test evaluation only. Model not serialized; duration prediction not started.")


MODEL SELECTION COMPLETE: confirmed
NO FURTHER TUNING: confirmed
FINAL TEST WILL BE EVALUATED ONCE ONLY: confirmed
Conditional assumption: estimated route distance is available before trip start.
Train rows used in final fit: 445000
Validation rows used in final fit: 97500
Total final-training rows: 542500
Final training time: 9.949 seconds
| Dataset | MAE | RMSE | R2 | Rows |
|---|---:|---:|---:|---:|
| Validation | 5.737909 | 10.986343 | 0.643793 | 6,814,901 |
| Final Test | 5.571799 | 10.324520 | 0.676153 | 6,730,257 |
Test minus validation differences:
MAE: -0.166110
RMSE: -0.661823
R2: +0.032360
STEP11_JSON={"train_sample_rows": 445000, "validation_sample_rows": 97500, "total_training_rows": 542500, "training_time_seconds": 9.94894320005551, "test_rows": 6730257, "test_metrics": {"MAE": 5.571799499757035, "RMSE": 10.324519986471309, "R2": 0.6761531056182961}, "validation_metrics": {"MAE": 5.737909, "RMSE": 10.986343, "R2": 0.643793}, "test_minus_validation": {"MAE": -0.16610950024

### Step 11 interpretation

| Dataset | MAE | RMSE | R2 | Rows |
|---|---:|---:|---:|---:|
| Validation | 5.737909 | 10.986343 | 0.643793 | 6,814,901 |
| Final test | 5.571799 | 10.324520 | 0.676153 | 6,730,257 |

Test minus validation is **-0.166110 MAE**, **-0.661823 RMSE**, and **+0.032360 R2**. Error therefore improved in the later chronological test period: MAE decreased by **2.895%**, RMSE decreased by **6.024%**, and R2 increased. Under the predeclared reporting heuristic (absolute MAE/RMSE change no more than 10% and absolute R2 change no more than 0.05), chronological generalization is stable.

This comparison is descriptive rather than a perfectly paired experiment. The validation metrics came from the earlier model fitted on the 445,000-row train sample; the final test model was refitted on **445,000 train rows plus 97,500 validation rows**, totaling **542,500 sampled rows**. The modest improvement can reflect both the later period and the additional training sample. It does not authorize any post-test change.

The selected model remains conditional-distance `LinearRegression` with `pickup_hour`, `month`, `distance_miles`, `provider_code`, `day_of_week`, `weekend`, `origin_loc_id`, and `dest_loc_id`. Its main limitation remains unchanged: `distance_miles` must represent an estimated route distance available before departure. Completed-trip mileage is not a valid pre-pickup input, and performance with a real route estimator may differ.

**PASS with conditional-feature warning.** The test split was opened once and streamed once for this final evaluation. The saved evaluation cell is locked against accidental reruns. No model, feature, preprocessing, cleaning, or split changes were made after seeing test results; no alternative model was tried, no model was serialized, and duration prediction was not started. **Stop after Step 11.**

## Step 12: Fare handover and serialization status

The fare-prediction work is finalized for documentation using the validation and single final-test results already recorded. No retraining, tuning, feature changes, cleaning changes, split changes, or test evaluation occurred in this step.

### Final model specification

- **Model:** default `LinearRegression` inside a scikit-learn `Pipeline`.
- **Numeric features:** `pickup_hour`, `month`, `distance_miles`.
- **Categorical features:** `provider_code`, `day_of_week`, `weekend`, `origin_loc_id`, `dest_loc_id`.
- **Target:** `base_fare`.
- **Preprocessing:** median numeric imputation; most-frequent categorical imputation; `OneHotEncoder(handle_unknown="ignore")`; `ColumnTransformer` inside the pipeline.
- **Final-training sample:** 445,000 train rows plus 97,500 validation rows, totaling 542,500 rows.

| Dataset | MAE | RMSE | R² | Rows |
|---|---:|---:|---:|---:|
| Validation | 5.737909 | 10.986343 | 0.643793 | 6,814,901 |
| Final test | 5.571799 | 10.324520 | 0.676153 | 6,730,257 |

LinearRegression was selected because the conditional-distance configuration achieved the best validation MAE, RMSE, and R² among the completed candidates. Without distance, LinearRegression (MAE 8.689478) still beat RandomForest (9.209341) and Extra Trees (9.106238) on MAE, while also remaining simpler. The route-ID experiment stopped before model fitting because one-hot encoding produced 22,392 columns. Adding `distance_miles` reduced validation MAE by 33.967% and RMSE by 21.122% compared with the strict seven-feature LinearRegression.

The selected configuration is conditional: **`distance_miles` must be an estimated route distance available before departure.** Actual completed-trip mileage must not be treated as known before pickup. When a reliable pre-trip estimate is unavailable, use the strict seven-feature configuration.

The test split was used once for final evaluation. The saved Step 11 cell remains locked, and no post-test model changes were made.

### Serialization completed and verified

The final serialized pipeline was recreated deterministically from the documented 542,500-row train-plus-validation sample because the fitted in-memory Step 11 object was no longer available. The test set was not accessed during serialization.

The serialization-only fit reused the exact Step 11 logic: up to 2,500 evenly spaced positions from every train and validation Parquet row group using `np.linspace(..., dtype=np.int64)`. Counts matched before fitting: **445,000 train rows**, **97,500 validation rows**, and **542,500 total rows**. Fitting took **9.559 seconds**.

The fitted pipeline was saved to `models/fare_pipeline.pkl` using `joblib`, with a size of **9,580 bytes**. `joblib.load()` returned a scikit-learn `Pipeline` containing the named steps `preprocessor` and `model`; the latter is `LinearRegression`. Five training-sample predictions were numeric, finite, and identical before and after serialization: `45.2050`, `48.7981`, `10.7151`, `12.8112`, and `42.7242` fare units.

**Status: PASS.** The model artifact, reload verification, non-test sanity check, handover, notebook, and figures are complete. No test data was accessed, no test metric was recomputed, and no tuning, feature, preprocessing, cleaning, split, or Member 1 data changes were made during serialization. Stop after Step 12.
